# Data Structure Analysis

Notebook này phân tích cấu trúc của file JSON pháp lý.
Nó tự tìm file dữ liệu theo nhiều vị trí để tránh lỗi đường dẫn.

In [6]:
import json
from pathlib import Path
from collections import Counter, defaultdict
from pprint import pprint

In [7]:
import os

DATA_FILENAME = 'tax_fee_legal_documents.json'
ENV_PATH = os.getenv('LEGAL_DATA_PATH')

candidate_paths = []
if ENV_PATH:
    candidate_paths.append(Path(ENV_PATH))

candidate_paths.extend([
    Path('backend/data') / DATA_FILENAME,
    Path('legal-chatbot/backend/data') / DATA_FILENAME,
])

resolved_path = None
for p in candidate_paths:
    if p.exists():
        resolved_path = p.resolve()
        break

if resolved_path is None:
    for base in [Path.cwd(), *Path.cwd().parents]:
        cand = base / 'backend' / 'data' / DATA_FILENAME
        if cand.exists():
            resolved_path = cand.resolve()
            break

if resolved_path is None:
    matches = list(Path.cwd().glob(f'**/{DATA_FILENAME}'))
    if matches:
        resolved_path = matches[0].resolve()

if resolved_path is None:
    print('Khong tim thay file du lieu.')
    print(f'Current working directory: {Path.cwd()}')
    print('Neu ban dang chay kernel remote, hay doi sang kernel local cua VS Code.')
    print('Hoac dat bien moi truong LEGAL_DATA_PATH den duong dan file JSON.')
    data = []
else:
    with resolved_path.open('r', encoding='utf-8') as f:
        data = json.load(f)
    print(f'Loaded file: {resolved_path}')

print(f'Root type: {type(data).__name__}')

Loaded file: C:\Users\ndtha\OneDrive\Desktop\app_luat\legal-chatbot\backend\data\tax_fee_legal_documents.json
Root type: list


In [3]:
if isinstance(data, list):
    print(f'Number of records: {len(data)}')
    if data:
        print('First record type:', type(data[0]).__name__)
elif isinstance(data, dict):
    print(f'Number of top-level keys: {len(data)}')
    print('Top-level keys:', list(data.keys())[:30])
else:
    print('Unsupported root structure for key analysis')

Number of records: 24570
First record type: dict


In [10]:
def summarize_structure(obj, depth=0, max_depth=2):
    if depth > max_depth:
        return {'type': type(obj).__name__, 'truncated': True}

    if isinstance(obj, dict):
        return {
            'type': 'dict',
            'keys': list(obj.keys()),
            'children': {k: summarize_structure(v, depth + 1, max_depth) for k, v in obj.items()},
        }

    if isinstance(obj, list):
        sample = obj[:3]
        return {
            'type': 'list',
            'length': len(obj),
            'sample_item_types': sorted({type(x).__name__ for x in sample}),
            'sample_children': [summarize_structure(x, depth + 1, max_depth) for x in sample],
        }

    return {'type': type(obj).__name__, 'value_sample': repr(obj)[:80]}

pprint(summarize_structure(data))

{'length': 24570,
 'sample_children': [{'children': {'content': {'children': {'content': {'truncated': True,
                                                                        'type': 'str'},
                                                            'id': {'truncated': True,
                                                                   'type': 'int'}},
                                               'keys': ['id', 'content'],
                                               'type': 'dict'},
                                   'dataset': {'type': 'str',
                                               'value_sample': "'th1nhng0/vietnamese-legal-documents'"},
                                   'metadata': {'children': {'document_number': {'truncated': True,
                                                                                 'type': 'str'},
                                                             'id': {'truncated': True,
                                            

In [9]:
if isinstance(data, list) and data and all(isinstance(x, dict) for x in data):
    key_freq = Counter()
    key_types = defaultdict(Counter)

    for item in data:
        for k, v in item.items():
            key_freq[k] += 1
            key_types[k][type(v).__name__] += 1

    print('Top 20 keys by frequency:')
    for k, c in key_freq.most_common(20):
        print(f'- {k}: {c}')

    print('\nValue types by key (top 10 keys):')
    for k, _ in key_freq.most_common(10):
        print(f'- {k}: {dict(key_types[k])}')
else:
    print('Dataset is not a list of dictionaries; skip key-frequency analysis.')

Top 20 keys by frequency:
- source_id: 24570
- content: 24570
- dataset: 24570
- metadata: 24570
- row_index: 24570
- source: 24570
- split: 24570
- target_bot: 24570

Value types by key (top 10 keys):
- source_id: {'str': 24570}
- content: {'dict': 24570}
- dataset: {'str': 24570}
- metadata: {'dict': 24570}
- row_index: {'int': 24570}
- source: {'str': 24570}
- split: {'str': 24570}
- target_bot: {'str': 24570}


## Bang Phan Tich Do Dai Va The Loai

Phan nay tao cac bang:
- Thong ke do dai van ban theo tung cot text.
- Phan bo the loai.
- Do dai trung binh theo the loai (neu tim duoc cot text va cot the loai).

In [11]:
from statistics import median

records = data if isinstance(data, list) else []

def get_nested(obj, path):
    cur = obj
    for key in path:
        if not isinstance(cur, dict) or key not in cur:
            return None
        cur = cur[key]
    return cur

def pick_nested_field(candidates, sample_record):
    for path in candidates:
        if get_nested(sample_record, path) is not None:
            return path
    return None

if not records or not isinstance(records[0], dict):
    print('Khong co du lieu dang list[dict] de tao bang phan tich.')
else:
    sample_record = records[0]

    text_candidates = [
        ('content', 'content'),
        ('metadata', 'title'),
    ]
    genre_candidates = [
        ('metadata', 'legal_type'),
        ('metadata', 'legal_sectors'),
        ('dataset',),
        ('target_bot',),
    ]

    text_path = pick_nested_field(text_candidates, sample_record)
    genre_path = pick_nested_field(genre_candidates, sample_record)

    print(f'Text field duoc chon: {text_path}')
    print(f'Genre field duoc chon: {genre_path}')

    length_rows = []
    fields_for_length = []
    if text_path:
        fields_for_length.append(('main_text', text_path))
    fields_for_length.append(('title', ('metadata', 'title')))

    for field_name, field_path in fields_for_length:
        lengths = []
        for r in records:
            value = get_nested(r, field_path)
            if isinstance(value, str):
                lengths.append(len(value.strip()))
        if lengths:
            length_rows.append({
                'field': field_name,
                'count': len(lengths),
                'min_len': min(lengths),
                'median_len': int(median(lengths)),
                'mean_len': round(sum(lengths) / len(lengths), 2),
                'max_len': max(lengths),
            })

    genre_rows = []
    genre_counts = {}
    if genre_path:
        for r in records:
            g = get_nested(r, genre_path)
            if isinstance(g, list):
                for item in g:
                    key = str(item).strip()
                    if key:
                        genre_counts[key] = genre_counts.get(key, 0) + 1
            elif g is not None:
                key = str(g).strip()
                if key:
                    genre_counts[key] = genre_counts.get(key, 0) + 1

        genre_rows = sorted(
            ({'genre': k, 'count': v} for k, v in genre_counts.items()),
            key=lambda x: x['count'],
            reverse=True,
        )

    by_genre_length_rows = []
    if text_path and genre_path and genre_counts:
        buckets = {}
        for r in records:
            text_value = get_nested(r, text_path)
            genre_value = get_nested(r, genre_path)
            if not isinstance(text_value, str):
                continue
            text_len = len(text_value.strip())

            if isinstance(genre_value, list):
                targets = [str(x).strip() for x in genre_value if str(x).strip()]
            elif genre_value is not None:
                g = str(genre_value).strip()
                targets = [g] if g else []
            else:
                targets = []

            if not targets:
                targets = ['UNKNOWN']

            for g in targets:
                buckets.setdefault(g, []).append(text_len)

        for g, vals in buckets.items():
            by_genre_length_rows.append({
                'genre': g,
                'count': len(vals),
                'mean_len': round(sum(vals) / len(vals), 2),
                'median_len': int(median(vals)),
            })

        by_genre_length_rows = sorted(by_genre_length_rows, key=lambda x: x['count'], reverse=True)

    try:
        import pandas as pd

        print('\nBang thong ke do dai:')
        display(pd.DataFrame(length_rows))

        print('\nBang phan bo the loai (top 30):')
        display(pd.DataFrame(genre_rows).head(30))

        print('\nBang do dai theo the loai (top 30):')
        display(pd.DataFrame(by_genre_length_rows).head(30))
    except Exception:
        print('\nBang thong ke do dai:')
        for row in length_rows:
            print(row)

        print('\nBang phan bo the loai (top 30):')
        for row in genre_rows[:30]:
            print(row)

        print('\nBang do dai theo the loai (top 30):')
        for row in by_genre_length_rows[:30]:
            print(row)

Text field duoc chon: ('content', 'content')
Genre field duoc chon: ('metadata', 'legal_type')

Bang thong ke do dai:


,field,count,min_len,median_len,mean_len,max_len
0,main_text,24570,537,2846,5907.12,1079065
1,title,24570,22,106,114.89,499



Bang phan bo the loai (top 30):


,genre,count
0,Công văn,20025
1,Quyết định,2422
2,Nghị quyết,1096
3,Thông tư,428
4,Nghị định,135
5,Chỉ thị,134
6,Văn bản hợp nhất,90
7,Thông báo,59
8,Điều ước quốc tế,56
9,Luật,22



Bang do dai theo the loai (top 30):


,genre,count,mean_len,median_len
0,Công văn,20025,3349.74,2537
1,Quyết định,2422,16138.58,4628
2,Nghị quyết,1096,10077.86,5254
3,Thông tư,428,25054.28,8227
4,Nghị định,135,20650.28,7223
5,Chỉ thị,134,7582.23,6967
6,Văn bản hợp nhất,90,92820.28,49803
7,Thông báo,59,8193.69,3760
8,Điều ước quốc tế,56,55213.30,50782
9,Luật,22,35661.18,17300
